# Finbert for Sentiment Score

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

/Users/visheshgupta/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
gold_dataset_news = pd.read_csv("../data/raw/gold_dataset_news.csv")

In [3]:
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
finbert = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

Device set to use mps:0


In [4]:
news_subset_df = gold_dataset_news[["Dates", "News"]]
news_subset_df

,Dates,News
0,28-01-2016,"april gold down 20 cents to settle at $1,116.1..."
1,13-09-2017,gold suffers third straight daily decline
2,26-07-2016,Gold futures edge up after two-session decline
3,28-02-2018,dent research : is gold's day in the sun comin...
4,06-09-2017,"Gold snaps three-day rally as Trump, lawmakers..."
...,...,...
10565,07-01-2013,gold seen falling from 3-week high this week
10566,27-09-2018,dominic frisby : now looks like a good time to...
10567,03-03-2017,Gold heading for worst week since November on ...
10568,11-06-2008,august gold up $7.60 at $878.80 an ounce on nymex


In [5]:
news_subset_df["Dates"] = pd.to_datetime(
    news_subset_df["Dates"], format="mixed", dayfirst=True, errors="coerce"
)
news_subset_df

/var/folders/mc/2wjfdchj6vsffbrpfbfgqw4w0000gn/T/ipykernel_45508/751433941.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  news_subset_df["Dates"] = pd.to_datetime(


,Dates,News
0,2016-01-28,"april gold down 20 cents to settle at $1,116.1..."
1,2017-09-13,gold suffers third straight daily decline
2,2016-07-26,Gold futures edge up after two-session decline
3,2018-02-28,dent research : is gold's day in the sun comin...
4,2017-09-06,"Gold snaps three-day rally as Trump, lawmakers..."
...,...,...
10565,2013-01-07,gold seen falling from 3-week high this week
10566,2018-09-27,dominic frisby : now looks like a good time to...
10567,2017-03-03,Gold heading for worst week since November on ...
10568,2008-06-11,august gold up $7.60 at $878.80 an ounce on nymex


In [6]:
grouped = gold_dataset_news.groupby("Dates")["News"].apply(list)
grouped

Dates
01-01-2007                [ncdex to launch 10 gm gold contract]
01-01-2009            [gold sparkles despite its year-end blip]
01-01-2015    [China's large imports lend solid support to g...
01-01-2016    [gold begins 2016 with a glitter on jewellers ...
01-01-2018    [gold's 2018 high price forecast: conservative...
                                    ...                        
31-12-2011    [gold's 'death cross' signals more losses comi...
31-12-2012    [Gold futures up at Rs 31,226 per 10 grams, go...
31-12-2013           [Gold futures down at Rs 27,620 per 10 gm]
31-12-2014        [Gold futures decline to Rs 26,931 per 10 gm]
31-12-2015    [Gold little changed in New Year's Eve trade, ...
Name: News, Length: 3761, dtype: object

In [7]:
from torch.nn.functional import softmax

In [ ]:
daily_sentiments = []

for date, news_list in tqdm(grouped.items(), desc="Computing FinBERT sentiment"):
    logits_list = []

    for sentence in news_list:
        inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
        inputs = {k: v.to("cpu") for k, v in inputs.items()}
        model.to("cpu")
        with torch.no_grad():
            logits = model(**inputs).logits.squeeze()
        logits_list.append(logits)

    stacked_logits = torch.stack(logits_list)
    avg_logits = torch.mean(stacked_logits, dim=0)
    probabilities = softmax(avg_logits, dim=0)

    labels = model.config.id2label
    final_probs = {labels[i]: float(probabilities[i]) for i in range(len(labels))}
    final_sentiment = max(final_probs, key=final_probs.get)

    daily_sentiments.append(
        {
            "Date": date,
            "Positive": final_probs.get("positive", 0.0),
            "Neutral": final_probs.get("neutral", 0.0),
            "Negative": final_probs.get("negative", 0.0),
            "Final Sentiment": final_sentiment,
        }
    )

Computing FinBERT sentiment: 3149it [06:50,  4.74it/s]

In [ ]:
sentiment_group_df = pd.DataFrame(daily_sentiments)

In [ ]:
sentiment_group_df["Date"] = pd.to_datetime(
    sentiment_group_df["Date"], dayfirst=True, errors="coerce"
)
sentiment_group_df = sentiment_group_df.sort_values("Date").reset_index(drop=True)

In [ ]:
sentiment_group_df[sentiment_group_df["Date"].isna() != True].reset_index(
    drop=True
).to_csv("../data/staged/Final_sentiment_score.csv")